# Predictive AI Evaluation Challenge — Metadata-only Latent-Factor Model (Colab)

This notebook clones the competition repo, downloads the public HuggingFace response parquets, trains a metadata-only PyTorch latent-factor model, and runs the official-like validation harness across 3 seeds.

**Statistical form**
$$\eta_{m,b,c} = \mu + a_m + b_{b,c} + \frac{u_m \cdot v_{b,c}}{\sqrt{k}}$$
$$p_{m,b,c} = \sigma(\eta_{m,b,c})$$

The model intentionally **does not see `item_content`**. Headline baseline to beat: a no-cross-term logistic regression with mean log-likelihood $\approx -0.5224$.

**Recommended runtime**: A100 (Runtime → Change runtime type → GPU → A100). L4/T4 also work — the model is small and the bottleneck is data movement.

## 1. Environment + GPU info

In [ ]:
import os, sys, subprocess, json, time, shutil
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'GPU memory: free={free/1e9:.2f}GB total={total/1e9:.2f}GB')
except ImportError:
    print('torch is not installed yet — will install in next cell.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Install dependencies

Colab already ships `torch`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`. We only need to ensure `huggingface_hub` and `datasets` are present for the data download. We also install `tqdm` (already there but pinned for safety).

In [ ]:
%pip install -q --upgrade huggingface_hub datasets pyarrow pandas numpy scikit-learn tqdm

## 3. Clone the competition repo

The repo bundles `validation_harness/`, `starting_kit/Model_Info/model_info.csv`, `starting_kit/benchmark_info/benchmark_info.csv`, and `Google_Collab_harness/` (this folder). It does **not** include the response parquets — those come from HuggingFace in the next step.

In [ ]:
REPO_URL = 'https://github.com/bwathomas/Prediction-Competition-321M.git'
REPO_DIR = Path('/content/Prediction-Competition-321M').resolve()
if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists, pulling latest …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

for sub in ['validation_harness', 'starting_kit/Model_Info', 'starting_kit/benchmark_info', 'Google_Collab_harness']:
    p = REPO_DIR / sub
    print(f'  {sub:40s} {"OK" if p.exists() else "MISSING"}')

GCH = REPO_DIR / 'Google_Collab_harness'
if str(GCH) not in sys.path:
    sys.path.insert(0, str(GCH))
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 4. Download the response parquets from HuggingFace

We download all `*.parquet` files from `aims-foundations/measurement-db` except the `*_traces.parquet` ones (different schema, not used). Total ≈ 1.5 GB.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = 'aims-foundations/measurement-db'
DATA_DIR = REPO_DIR / 'starting_kit' / 'Data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type='dataset')
wanted = [
    f for f in files
    if f.endswith('.parquet') and not f.endswith('_traces.parquet')
]
print(f'{len(wanted)} parquets to download')

for f in wanted:
    out = DATA_DIR / Path(f).name
    if out.exists() and out.stat().st_size > 0:
        continue
    print(f'  downloading {f} …', flush=True)
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset', local_dir=str(DATA_DIR), local_dir_use_symlinks=False)
    if Path(p).resolve() != out.resolve():
        try:
            shutil.copy2(p, out)
        except shutil.SameFileError:
            pass
print('done. files:')
subprocess.run(['ls', '-lh', str(DATA_DIR)], check=False)

## 5. Build the official item-cold-start split

Reuses `validation_harness/scripts/prepare_split.py` so the validation we report is exactly the official-like protocol.

In [ ]:
HARNESS_DIR = REPO_DIR / 'validation_harness'
SPLITS_DIR = HARNESS_DIR / 'splits' / 'v1'
if not (SPLITS_DIR / 'train.parquet').exists():
    subprocess.check_call([
        sys.executable,
        str(HARNESS_DIR / 'scripts' / 'prepare_split.py'),
        '--data-dir', str(DATA_DIR),
        '--out-dir',  str(SPLITS_DIR),
        '--val-fraction', '0.10',
        '--seed', '0',
    ])
else:
    print(f'Reusing existing split at {SPLITS_DIR}')
subprocess.run(['ls', '-lh', str(SPLITS_DIR)], check=False)

## 6. Train the latent-factor model + run official-like validation

This wraps everything: fit preprocessor on TRAIN ONLY, aggregate by (model, benchmark, condition) cells, train with AdamW + AMP + early stopping, package a submission folder, and run `validation_harness/scripts/run_validation.py` across seeds 0/1/2.

Defaults: `latent_dim=16`, `hidden_dim=256`, `num_layers=2`, `dropout=0.1`, `batch_size=65536`, `epochs=30`, `patience=5`.

**Live progress reporting**

- A live `tqdm` bar tracks the planned **total training steps** (= `epochs * steps_per_epoch`) with continuously updated **% done** and **ETA**. The bar's postfix shows the current `epoch`, the current batch `loss`, and the best validation log-likelihood seen so far (`best_val`).
- Every **10 optimizer steps** (`--log-every-steps 10`) a one-line summary is printed: `step S/Total (X% done with training, time estimated is Y)  epoch=…  loss=…  elapsed=…`.
- At the end of every **epoch** a per-epoch summary is logged (`train_loss`, `val_ll`, `val_brier`, `val_auc`, `epoch_seconds`, `eta`, current best with epoch). A trailing `*` marks an improvement.

In [ ]:
import os, subprocess

OUTPUT_DIR = REPO_DIR / 'outputs' / 'latent_factor'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', str(GCH / 'run_latent_factor_colab.py'),
    '--data-dir',                str(DATA_DIR),
    '--splits-dir',              str(SPLITS_DIR),
    '--model-info-csv',          str(REPO_DIR / 'starting_kit' / 'Model_Info' / 'model_info.csv'),
    '--benchmark-info-csv',      str(REPO_DIR / 'starting_kit' / 'benchmark_info' / 'benchmark_info.csv'),
    '--validation-harness-dir',  str(HARNESS_DIR),
    '--output-dir',              str(OUTPUT_DIR),
    '--latent-dim', '16',
    '--hidden-dim', '256',
    '--num-layers', '2',
    '--dropout',    '0.1',
    '--weight-decay','1e-4',
    '--batch-size', '65536',
    '--epochs',     '30',
    '--patience',   '5',
    # --- live progress: per-step loss + tqdm-style "X% done, ETA Y" bar ---
    '--log-every-steps', '10',     # one-line training-step log every 10 optimizer steps
    '--progress-bar',              # tqdm bar over total training steps with live ETA
    # --- official-like validation across 3 seeds ---
    '--official-seeds', '0', '1', '2',
    '--official-n', '5000',
    '--official-k', '5',
    '--logistic-baseline-ll', '-0.5224',
]


def _run_streaming(argv):
    """Run a child process and stream its stdout/stderr into the cell.

    Plain ``subprocess.run(cmd)`` lets the child write to the kernel's raw
    stdout/stderr file descriptors, which IPython does NOT forward to the
    cell display in most Colab/Jupyter setups -- so the cell looks empty
    even though the script printed plenty. We capture stdout (with stderr
    merged in) and re-print line-by-line so IPython's OutStream picks it up.
    """
    print('Running:', ' '.join(argv), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    proc = subprocess.Popen(
        argv,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge so order is preserved
        bufsize=1,                 # line-buffered
        text=True,
        env=env,
    )
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    finally:
        proc.stdout.close()
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, argv)


_run_streaming(cmd)

## 7. (Optional) sequential hyperparameter sweep with full progress

Runs the same script with `--sweep`, which samples `--sweep-budget` configs from a 7-dimensional grid:

| dim | values |
| --- | --- |
| `latent_dim`   | 4, 8, 16, 32 |
| `hidden_dim`   | 128, 256 |
| `dropout`      | 0.05, 0.1, 0.2 |
| `weight_decay` | 1e-4, 1e-3 |
| `lr`           | 1e-3, 3e-3 |
| `id_emb_l2`    | 1e-4, 1e-3 |
| `patience`     | 5, 10 |

(`--sweep-mode full` would walk the entire 384-config grid; `random` picks a uniform subset of size `--sweep-budget`.)

**Why sequential here?** With `--parallel-runs 1` we get the *exact same* per-step progress as section 6 for **every** sweep run, one at a time. After each run finishes the next one starts and prints its own tqdm bar + per-step loss, so the cell output reads like a clean diary of "run 1 done → starting run 2 → ...". This is much easier to monitor than 8 interleaved tqdm bars on the same GPU. The shared preprocessor + tensor cache (built once in section 6) keeps per-run startup cheap.

If you'd rather trade observability for raw throughput, set `--parallel-runs 0` (auto) or e.g. `--parallel-runs 8` and the script will dispatch the configs through `ProcessPoolExecutor` with the `spawn` start method (tqdm bars are auto-disabled in that mode to avoid mangled output, but per-step + per-epoch text logs still go through).

In [ ]:
sweep_cmd = cmd + [
    '--sweep',
    '--sweep-mode', 'random',
    '--sweep-budget', '24',
    # Sequential: one process at a time so each run prints a clean tqdm bar +
    # per-10-step loss line, then we "pick up" the next run when it finishes.
    # Bump to e.g. 8 for parallel throughput (per-step bar is auto-disabled then).
    '--parallel-runs', '1',
    '--log-every-steps', '10',
    '--progress-bar',
    '--amp',
]
print('Running:', ' '.join(sweep_cmd))
subprocess.run(sweep_cmd, check=True)

## 8. Read the metrics + interpret results

In [ ]:
import pandas as pd, json
summary = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
print(json.dumps(summary, indent=2))

print('\nbaseline_comparison.csv:')
print(pd.read_csv(OUTPUT_DIR / 'baseline_comparison.csv').to_string(index=False))

print('\nofficial_seeds.csv:')
print(pd.read_csv(OUTPUT_DIR / 'official_seeds.csv').to_string(index=False))

if (OUTPUT_DIR / 'runs.csv').exists():
    print('\nruns.csv (top 10 by final_val_log_likelihood):')
    runs = pd.read_csv(OUTPUT_DIR / 'runs.csv')
    cols = ['run_idx', 'latent_dim', 'weight_decay', 'dropout', 'best_epoch',
            'final_train_log_likelihood', 'final_val_log_likelihood', 'wall_seconds']
    cols = [c for c in cols if c in runs.columns]
    print(runs[cols].sort_values('final_val_log_likelihood', ascending=False).head(10).to_string(index=False))

off_mean = summary.get('official_mean_log_likelihood')
imp = summary.get('improvement_vs_logistic_baseline')
if off_mean is not None and imp is not None:
    verdict = 'BEATS' if (imp or 0) > 0 else 'TRAILS'
    print(f'\nOfficial-like mean LL = {off_mean:+.4f}    {verdict} the logistic baseline by {imp:+.4f}')

## 9. Download the best model

Bundles the best run's artifacts (`best_model.pt`, raw `weights.pt`, fitted `preprocessor.pkl`, metadata CSVs, the runtime-ready `submission/` folder, and the `reproduce.json` / `reproduce.sh` / `reproduce.bat` manifest) into a single `latent_factor_submission.zip` and triggers a browser download via `google.colab.files`. Outside Colab the cell just prints the absolute path so you can copy it manually.

After downloading you can either:

- unzip and resubmit `submission/` to the official platform, or
- reload the model elsewhere with:

  ```python
  from latent_factor_pytorch import load_artifacts
  model, preprocessor, config = load_artifacts('latent_factor_submission/')
  ```

In [ ]:
import shutil, zipfile
from pathlib import Path

# Files at the top level of OUTPUT_DIR that are useful to ship.
TOP_LEVEL_FILES = [
    'best_model.pt',           # full bundle: state_dict + config
    'weights.pt',               # raw state_dict only (smaller)
    'preprocessor.pkl',         # fitted preprocessor (vocabs, scalers, lookups)
    'model_info.csv',           # baked metadata for offline reload
    'benchmark_info.csv',       # baked metadata for offline reload
    'metrics.json',             # final scalar metrics
    'baseline_comparison.csv',  # vs. constant_0.5 / base-rate / logistic
    'official_seeds.csv',       # per-seed official-like LL
    'runs.csv',                 # all sweep configs + their final metrics
    'reproduce.json',           # exhaustive manifest (env, git, hashes, args)
    'reproduce.sh',             # one-line CLI to retrain best (Linux/Colab)
    'reproduce.bat',            # one-line CLI to retrain best (Windows)
]

# Resolve OUTPUT_DIR if it wasn't already defined (e.g. running this cell standalone)
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path('/content/Prediction-Competition-321M/outputs/latent_factor').resolve()

OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
SUBMISSION_DIR = OUTPUT_DIR / 'submission'
ZIP_PATH = OUTPUT_DIR / 'latent_factor_submission.zip'
print(f'Bundling artifacts under: {OUTPUT_DIR}')

if not OUTPUT_DIR.exists():
    raise SystemExit(f'OUTPUT_DIR does not exist: {OUTPUT_DIR}. Run section 6 first.')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    bundled, skipped = [], []
    for fname in TOP_LEVEL_FILES:
        src = OUTPUT_DIR / fname
        if src.exists():
            zf.write(src, arcname=f'latent_factor_submission/{fname}')
            bundled.append(fname)
        else:
            skipped.append(fname)
    if SUBMISSION_DIR.exists():
        for p in SUBMISSION_DIR.rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts:
                arc = Path('latent_factor_submission/submission') / p.relative_to(SUBMISSION_DIR)
                zf.write(p, arcname=str(arc).replace('\\', '/'))

zip_mb = ZIP_PATH.stat().st_size / 1e6
print(f'  bundled : {bundled}')
if skipped:
    print(f'  skipped : {skipped}  (these files were not produced — check section 6 ran cleanly)')
if SUBMISSION_DIR.exists():
    print(f'  + full submission/ folder ({sum(1 for p in SUBMISSION_DIR.rglob("*") if p.is_file())} files)')
print(f'  zip     : {ZIP_PATH}  ({zip_mb:.2f} MB)')

# Trigger a browser download in Colab; fall back gracefully elsewhere.
try:
    from google.colab import files as _colab_files  # type: ignore
    print(f'\nTriggering download of {ZIP_PATH.name} ...')
    _colab_files.download(str(ZIP_PATH))
    # Also offer the standalone state_dict for users who only want the weights.
    weights_pt = OUTPUT_DIR / 'weights.pt'
    if weights_pt.exists():
        print(f'Triggering download of {weights_pt.name} (raw state_dict only) ...')
        _colab_files.download(str(weights_pt))
except ImportError:
    print('\nNot running in Colab — no browser download triggered.')
    print(f'Copy this file off the machine yourself: {ZIP_PATH}')
    print(f'Standalone weights file:                  {OUTPUT_DIR / "weights.pt"}')
except Exception as e:
    print(f'\ngoogle.colab.files.download failed: {type(e).__name__}: {e}')
    print(f'Manual fallback path: {ZIP_PATH}')